# **Retreival part**

This part is didacted to test and create collection of vector database by chunking, and embedding the different documents of each court

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path

DB_PATH = Path(".").resolve().parent / "Collections"

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0+cu132
CUDA build: 13.2
CUDA available: True


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## **Chunking**

In this part we gonna use a a basic chunking strategy based on paragraphs by using fixed length and overlap

In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from Dataset.upload import get_data
from typing import Literal
import pandas as pd

### **Help functions for text cleaning**

In [5]:
import unicodedata
import re

In [6]:
#datalemu
def clear_text(text: str) -> str:
    # Normalize Unicode characters
    text = unicodedata.normalize("NFC", text)

    # Remove common page-break and control characters
    text = text.replace("\x0c", " ")
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # Remove spaces around hyphens used as line-wrap artifacts
    # (Avoid blindly joining words around every hyphen.)

    return text.strip()

### **Basic Chunk (simple, paragraph)**

In [7]:
def chunk_characters(text : str, overlap : int, chunk_size : int, chunk_type : Literal["recursive", "simple"] = "recursive") -> list : 
    
    if chunk_type == "simple" : 
        splitter = CharacterTextSplitter(
            separator="",
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )
    elif chunk_type == "recursive" : 
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
            length_function=len,
        )


    chunks = splitter.split_text(text)

    return chunks

## **Getting the data**

In [8]:
cour_d_appel = await get_data(juridication="cour_d_appel", length=100)

In [9]:
tribunal_judiciaire = await get_data(juridication="tribunal_judiciaire", length=100)

In [10]:
cour_de_cassation = await get_data(juridication="cour_de_cassation", length=100)

In [11]:
data = pd.concat(
    [cour_d_appel, tribunal_judiciaire, cour_de_cassation],
    ignore_index=True
)

In [12]:
data

,id,themes,text,location,numbers,jurisdiction,decision_date,summary
0,6253c84cbd3db21cbdd84d1c,"[protection des consommateurs, surendettement,...",COUR D'APPEL D'ANGERS \n\nChambre des \n\nREDR...,Cour d'appel d'Angers,[1995-00807],Cour d'appel,1996-03-25,None
1,6253c84cbd3db21cbdd84d1d,[bail a loyer (loi du 6 juillet 1989)],Par acte sous seing privé en date du 6 août 19...,Cour d'appel de Versailles,[1996-4998],Cour d'appel,1996-06-12,None
2,6253c84cbd3db21cbdd84d1f,"[conventions internationales, accords et conve...",FAITS ET PROCEDURE : La Société SCHLUMBERGER I...,Cour d'appel de Versailles,[1994-1455],Cour d'appel,1996-12-03,None
3,6253c84cbd3db21cbdd84d20,"[procedures civiles d'execution, mesures d'exé...","Dans ses écrits devant la Cour, Monsieur X... ...",Cour d'appel de Caen,[97-01175],Cour d'appel,1997-01-31,None
4,6253c84cbd3db21cbdd84d21,"[bail a loyer (loi du 6 juillet 1989), preneur...","Par acte sous seing privé du 15 mars 1990, la ...",Cour d'appel de Versailles,[1994-7957],Cour d'appel,1997-01-31,None
...,...,...,...,...,...,...,...,...
295,6079411c9ba5988459c40696,"[astreinte, liquidation, montant, appreciation...","Sur le moyen unique, en ses deux branches :\n\...",None,[57-10.110],Cour de cassation,1959-10-20,Est légalement justifié l'arrêt qui décide que...
296,6079b1c29ba5988459c533eb,"[1) contrat de travail, rupture par le salarié...","Sur les premier et deuxième moyens réunis, pri...",None,[57-41.001],Cour de cassation,1959-10-22,Est légalement justifiée l'allocation d'une in...
297,6079b1c29ba5988459c533be,"[travail reglementation, règlement intérieur, ...",Sur le moyen unique : Vu l'article 1134 du Cod...,None,[58-40.133],Cour de cassation,1959-11-06,Manque de base légale la décision réduisant d'...
298,6079b1c29ba5988459c533ec,"[1) contrat de travail, responsabilité du sala...",Sur le moyen unique et le premier moyen additi...,None,[57-40.656],Cour de cassation,1959-11-19,"Un employeur ne peut, en vertu d'un règlement ..."


## **Upload the docs to the collection**

In this section, we are going to create a collection where we can store our chunks documents. 
We are going to use Chromadb as a vector db to store the chunks.

To store the documents in a significant way, we need to : 
- Add metadata information for each chunk to keep the filter option available, and also to retreive the docs
- Add the chunks with their id but also with their source id of the document to distinct the bag of chunks

### **Chunk the documents**

In [13]:
def chunk_docs(
        data : pd.DataFrame,
        chunk_size : int = 300,
        chunk_overlap : int = 100
) :
    ids: list[str] = []
    documents: list[str] = []
    metadatas: list[dict] = []

    for row in data.itertuples(index = False):

        if pd.isna(row.text) or not str(row.text).strip():
            continue

        source_id = str(row.id)
        source_text = str(row.text)
        location = str(row.location) if row.location is not None else ""
        source_themes = row.themes
        decision_date = row.decision_date

        if isinstance(source_themes, (list, tuple, set)):
            themes_text = ", ".join(map(str, source_themes))
        elif source_themes is None or (
            isinstance(source_themes, float) and pd.isna(source_themes)
        ):
            themes_text = ""
        else : 
            themes_text = str(source_themes)

        #Chunking part :
        clean_text = clear_text(source_text)
        chunks = chunk_characters(
            text=clean_text,
            chunk_size=chunk_size,
            overlap=chunk_overlap
        )
        #Building the metadata
        numbers = ""
        
        for num in row.numbers :
            numbers += f"{str(num)}, "

        for chunk_index, chunk in enumerate(chunks):
            chunk = str(chunk).strip()

            if not chunk:
                continue
            # Building the chunk id
            chunk_id = f"{source_id}_chunk_{chunk_index}"

            ids.append(chunk_id)
            documents.append(chunk)
            metadatas.append(
                {
                    "source_id": source_id,
                    "juridiction" : row.jurisdiction,
                    "themes" : themes_text,
                    "location" : location, 
                    "Numéros" : numbers,
                    "Date de decision" : row.decision_date,
                    "Résumé" : row.summary if row.summary is not None else ""
                }
            )

    return documents, ids, metadatas


### **Embed the chunks**

In [14]:
#downloading the embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
def embed_docs(
        documents : list,
        ids : list,
        metadatas : list,
        batch_size : int,
        collection : chromadb.Collection
):
    #embed the documents
    try:
        for start in range(0, len(documents), batch_size):
                end = start + batch_size

                batch_documents = documents[start:end]
                batch_ids = ids[start:end]
                batch_metadatas = metadatas[start:end]

                embeddings = embedding_model.encode(
                    batch_documents,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                    batch_size=batch_size,
                    device=device
                )

                collection.add(
                    ids=batch_ids,
                    documents=batch_documents,
                    metadatas=batch_metadatas,
                    embeddings=embeddings.tolist(),
                )      
        print("The data are uploaded to the collection successfully ! ")
        return True
    except Exception:
        return False
    

### **Create collection** 

In [16]:
def create_and_upload_to_collection(
        batch_size : int,
        data : pd.DataFrame,
        chunk_overlap : int = 100,
        chunk_size : int = 300,
        collection_name : str = "jurisprudence",
):

    #Creating the collection
    client = chromadb.PersistentClient(path=str(DB_PATH))

    collection = client.get_or_create_collection(
        name=collection_name,
        configuration={
            "hnsw" : {
                "space":"cosine"
            }
        }
    )

    #Getting the data after chunking
    documents, ids, metadatas = chunk_docs(data = data, chunk_size=chunk_size, chunk_overlap = chunk_overlap)

    #Embedding the data and upload to the collection
    result = embed_docs(
        documents=documents,
        ids = ids,
        metadatas=metadatas,
        batch_size=batch_size,
        collection=collection
    )

    if result and collection.count() != 0:
        print(f"Uploaded {collection.count()} chunks into the the collection '{collection_name}'")
    else :
        print("An error has occured during the upload of the data")

    return collection



### **Final test (upload the chunks)**

In [17]:
collection_uploaded = create_and_upload_to_collection(
    batch_size=64,
    data=data,
    chunk_overlap=500,
    chunk_size=1000,
)

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.06it/s]


The data are uploaded to the collection successfully ! 
Uploaded 12370 chunks into the the collection 'jurisprudence'


In [23]:
chunk_sizes = [100, 300, 500]
overlap_sizes = [40, 140, 240]
collection_names = [
    'jurisprudence_chunk_100_overlap_40',
    'jurisprudence_chunk_300_overlap_140',
    'jurisprudence_chunk_500_overlap_240'
]


In [24]:
for chunk_size, overlap_size, collection_name in zip(chunk_sizes, overlap_sizes, collection_names):
    collection_uploaded = create_and_upload_to_collection(
        batch_size=100,
        data=data,
        chunk_overlap=overlap_size,
        chunk_size=chunk_size,
        collection_name=collection_name
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The data are uploaded to the collection successfully ! 
Uploaded 23946 chunks into the the collection 'jurisprudence_chunk_500_overlap_240'
